# RGB-only Agentic Memory 实战教程

本 notebook 讲解并演示完整运行时闭环：

```text
RGB -> LingBot depth + c2w + intrinsics -> local submap
    -> spatial memory -> VLM semantic objects -> Pi
    -> scene graph -> knowledge memory -> reasoning -> navigation
```

运行时目标是纯 RGB agent。LiDAR、GT depth 与 GT pose 仅用于离线误差研究，不是运行时依赖。

## 1. 环境与解释器

项目使用 `enum.StrEnum`，因此 notebook 必须使用项目 `.venv` 的 Python 3.12 kernel。已注册的 kernel 名称是 **AgenticMemoryNav Python 3.12**。如果本单元提示版本错误，请在 VS Code 右上角选择该 kernel 后重新运行。

In [ ]:
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError(
        'This workshop requires Python 3.11+. Select the AgenticMemoryNav Python 3.12 kernel.'
    )

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import numpy as np

print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('Project root:', project_root)

## 2. RGB-only local submap：稳定后才写入 spatial memory

LingBot 的 depth head 与 camera head 产生 depth、相机内参和 c2w。通过标准反投影得到每帧局部点云。

机器人移动会改变视野，因此不能用点云质心变化判断错误。系统使用相邻点云的对称最近邻 overlap residual：

$$
r(P_t,P_{t+1})=\frac{1}{2}(d(P_t,P_{t+1})+d(P_{t+1},P_t))
$$

窗口内最大残差低于阈值时，submap 被标为 stable。RGB-only policy 允许 stable submap 直接写入 spatial memory，并保留 confidence、residual 和 provenance。

In [ ]:
from agentic_memory_nav.common.types import MappingUpdate, Pose3D
from agentic_memory_nav.mapping.local_submap import LocalSubmapBuilder

def mapping_update(frame_index, offset_x):
    # 模拟由 depth + c2w 反投影的局部点云。
    cloud = np.array([
        [offset_x, 0.0, 1.0],
        [offset_x + 0.02, 0.0, 1.0],
        [offset_x, 0.02, 1.0],
    ], dtype=np.float32)
    return MappingUpdate(
        frame_id=f'frame_{frame_index:04d}',
        timestamp=float(frame_index),
        camera_pose=Pose3D(position=(offset_x, 0.0, 0.0)),
        depth=np.ones((2, 2), dtype=np.float32),
        confidence=np.ones((2, 2), dtype=np.float32),
        local_pointcloud=cloud,
        global_pointcloud=cloud,
        is_keyframe=True,
        map_version=frame_index + 1,
    )

builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
submap = None
for index, offset_x in enumerate((0.00, 0.03, 0.06)):
    submap = builder.add(mapping_update(index, offset_x))

assert submap is not None
assert submap.stable
print('stable:', submap.stable)
print('frame ids:', submap.frame_ids)
print('overlap residual (m):', round(submap.geometric_residual_m, 4))
print('submap points:', len(submap.points))

In [ ]:
unstable_builder = LocalSubmapBuilder(window_frames=3, frame_stride=1, stability_threshold_m=0.10)
unstable_builder.add(mapping_update(0, 0.00))
unstable_builder.add(mapping_update(1, 0.03))
unstable = unstable_builder.add(mapping_update(2, 2.00))

assert unstable is not None
assert not unstable.stable
print('stable:', unstable.stable)
print('overlap residual (m):', round(unstable.geometric_residual_m, 4))
print('memory decision: do not commit this geometry window')

## 3. 从 VLM 语义到物体点云 $P_i$

VLM 负责类别、属性、bbox 和三元组候选；depth+c2w 负责几何。最小可运行路径为：

```text
VLM bbox -> instance mask -> selected depth pixels -> backprojection -> Pi NPZ artifact
```

当前演示使用 deterministic bbox mask。未来可以替换为 SAM、Grounding-DINO + SAM 或开源点云实例分割模型，后续 $P_i$、graph、memory 的接口不变。

In [ ]:
from agentic_memory_nav.common.types import CameraIntrinsics, FrameObservation, ObjectObservation
from agentic_memory_nav.geometry.pointcloud_store import PointCloudStore
from agentic_memory_nav.perception.instance_segmentation import BoundingBoxSegmenter, InstanceGeometryEnricher

rgb = np.zeros((48, 64, 3), dtype=np.uint8)
depth = np.full((48, 64), 2.0, dtype=np.float32)
frame = FrameObservation(
    frame_id='pi_frame', timestamp=0.0, rgb=rgb, depth=depth,
    camera_intrinsics=CameraIntrinsics(60.0, 60.0, 32.0, 24.0, 64, 48),
    camera_pose=Pose3D(),
)
mapping = mapping_update(0, 0.0)
mapping.depth = depth
mapping.confidence = np.ones_like(depth)
cube_observation = ObjectObservation(
    observation_id='obs_red_cube', category='cube', attributes={'color': 'red'},
    bbox_2d=(20, 12, 44, 36), center_3d=(0.0, 0.0, 0.0),
    dimensions_3d=(0.0, 0.0, 0.0), confidence=0.9,
    timestamp=0.0, frame_id=frame.frame_id,
)

pi_store = PointCloudStore(Path('/tmp/agentic_memory_nav_workshop_pi'))
enricher = InstanceGeometryEnricher(BoundingBoxSegmenter(), pi_store)
cube = enricher.enrich(frame, mapping, [cube_observation])[0]

assert cube.geometry is not None
print('Pi artifact:', cube.geometry.artifact_path)
print('Pi points:', cube.geometry.point_count)
print('Pi centroid:', cube.geometry.centroid_3d)
print('Pi dimensions:', cube.geometry.dimensions_3d)

## 4. Scene Graph 变成 Knowledge Memory

对象 $P_i$ 和 room observation 进入 `SceneGraphUpdater` 后，会成为 object/room nodes。几何规则在 object 与 room 间生成 `inside` 等边。

`KnowledgeMemory.materialize(graph)` 再将 node 和 directed relation edge 变成 SQLite 中可检索的 memory graph facts。这样 agent 后续即使不再拥有当前 VLM 响应，也能查询历史证据。

In [ ]:
from agentic_memory_nav.memory.knowledge_memory import KnowledgeMemory
from agentic_memory_nav.memory.sqlite_store import SQLiteMemory
from agentic_memory_nav.scene_graph.graph import SceneGraph
from agentic_memory_nav.scene_graph.updater import SceneGraphUpdater

# room 与 cube 位置分开，association 不会错误把不同类别合并为同一 node。
room = ObjectObservation(
    observation_id="obs_kitchen",
    category="kitchen",
    attributes={"kind": "room"},
    bbox_2d=(0, 0, 64, 48),
    center_3d=(3.0, 0.0, 3.0),
    dimensions_3d=(6.0, 3.0, 6.0),
    confidence=0.99,
    timestamp=0.0,
    frame_id="pi_frame",
)
graph = SceneGraph()
SceneGraphUpdater(graph).update([room, cube])

memory_path = Path("/tmp/agentic_memory_nav_workshop_memory.sqlite3")
if memory_path.exists():
    memory_path.unlink()
memory = SQLiteMemory(memory_path)
knowledge = KnowledgeMemory(memory)
facts_created = knowledge.materialize(graph)

print("graph nodes:", [(node.label, node.node_type.value) for node in graph.nodes()])
print("graph relations:", [(edge.relation, round(edge.confidence, 2)) for edge in graph.edges()])
print("knowledge facts created:", facts_created)
print("memory graph query:", [item.content for item in knowledge.retrieve_subgraph("cube kitchen")])

## 5. 从 Memory Graph 推理到 Navigation

现在进入 agent 的决策部分：

1. parser 将自然语言任务转为结构化 goal；
2. `NativeReasoner` 从 graph 和 knowledge memory 验证目标与 room relation；
3. `RuleBasedPlanner` 根据 reasoning result 生成高层 action；
4. 有充分证据时是 `NAVIGATE`，没有目标证据时是 `EXPLORE`，关系不完整时会携带 information gap 并要求重规划/验证。

当前 MVP parser 可稳定识别 `Find cube in the kitchen`。颜色等细粒度语义仍保存在 VLM/graph 的 node attributes 中；生产系统可以替换为更强的结构化语言 parser。

In [ ]:
from agentic_memory_nav.planning.rule_based_fallback import RuleBasedPlanner
from agentic_memory_nav.planning.task_parser import RuleBasedTaskParser
from agentic_memory_nav.reasoning.native_reasoner import NativeReasoner

task = RuleBasedTaskParser().parse("Find cube in the kitchen")
reasoner = NativeReasoner(knowledge)
reasoning_result = reasoner.resolve(graph, task.parsed_goal)
planner = RuleBasedPlanner(approach_distance=0.6)
plan = planner.plan(
    task=task,
    robot_pose=Pose3D(position=(0.0, 0.0, 0.0)),
    graph=graph,
    memory=memory,
    replan_reason="new stable RGB-only submap",
)

print("parsed goal:", task.parsed_goal)
print("reasoning target:", reasoning_result.target_id)
print("reasoning evidence:", reasoning_result.evidence_ids)
print("action type:", plan.action.action_type.value)
print("navigation target:", plan.action.target)
print("waypoint:", plan.action.waypoint)
print("plan confidence:", round(plan.confidence, 3))
print("information gaps:", plan.information_gaps)

assert plan.action.action_type.value == "navigate"
assert plan.action.target == reasoning_result.target_id
assert plan.action.waypoint is not None

## 6. 实时 Reasoning：每个 RGB frame 都立刻返回 Navigation Action

上面的代码展示了“已有 graph/memory 后的一次 plan”。真实 agent 需要连续循环：

```text
new RGB frame
-> mapper 更新 depth/c2w 或 local submap
-> VLM / perception 输出当前语义 observation
-> Pi、Scene Graph、Knowledge Memory 更新
-> NativeReasoner 查询更新后的 memory graph
-> Planner 立即返回下一步 NAVIGATE / EXPLORE / VERIFY action
```

`RealtimeAgent.ingest_frame(frame)` 就是这个一步式 runtime API。外部控制器负责采集 RGB 和执行返回的 waypoint；agent 负责更新记忆并在每帧重规划。

In [ ]:
from agentic_memory_nav.agent.realtime_agent import RealtimeAgent


def realtime_frame(index):
    return FrameObservation(
        frame_id=f"realtime_{index:04d}",
        timestamp=float(index),
        rgb=np.zeros((64, 96, 3), dtype=np.uint8),
        depth=np.full((64, 96), 2.0, dtype=np.float32),
        camera_intrinsics=CameraIntrinsics(80.0, 80.0, 48.0, 32.0, 96, 64),
        camera_pose=Pose3D(),
        robot_pose=Pose3D(),
    )


realtime_root = Path("/tmp/agentic_memory_nav_realtime_demo")
agent = RealtimeAgent(realtime_root, "Find the red cup in the kitchen")
try:
    # 第一帧只有 kitchen；memory graph 没有 cup，因此实时 action 是 explore。
    first_decision = agent.ingest_frame(realtime_frame(0))
    print("frame 0 action:", first_decision.plan.action.action_type.value)
    print("frame 0 graph nodes:", first_decision.graph_nodes)

    # 第二帧出现 red cup；更新 graph/memory 后，同一任务立刻重规划为 navigate。
    second_decision = agent.ingest_frame(realtime_frame(1))
    print("frame 1 action:", second_decision.plan.action.action_type.value)
    print("frame 1 target:", second_decision.plan.action.target)
    print("frame 1 waypoint:", second_decision.plan.action.waypoint)
    print("frame 1 knowledge facts created:", second_decision.knowledge_facts_created)

    assert first_decision.plan.action.action_type.value == "explore"
    assert second_decision.plan.action.action_type.value == "navigate"
    assert second_decision.plan.action.waypoint is not None
finally:
    agent.close()

### 没有目标证据时：Explore，而不是编造位置

这是一条重要的 agent safety / epistemic policy：memory graph 中不存在目标，就去探索并采集新的 RGB observation，不把未知对象假装成已定位目标。

In [ ]:
empty_graph = SceneGraph()
empty_memory_path = Path('/tmp/agentic_memory_nav_workshop_empty.sqlite3')
if empty_memory_path.exists():
    empty_memory_path.unlink()
empty_memory = SQLiteMemory(empty_memory_path)

explore_plan = RuleBasedPlanner().plan(
    task=task,
    robot_pose=Pose3D(position=(0.0, 0.0, 0.0)),
    graph=empty_graph,
    memory=empty_memory,
)

print('action type:', explore_plan.action.action_type.value)
print('exploration waypoint:', explore_plan.action.waypoint)
print('information gaps:', explore_plan.information_gaps)
print('reason:', explore_plan.action.reason)

assert explore_plan.action.action_type.value == 'explore'
assert explore_plan.replan_required
empty_memory.close()
memory.close()

## 6. Runtime 与离线评测的边界

运行时只依赖 RGB：

```text
RGB -> LingBot depth/c2w -> stable local submap -> spatial memory
RGB -> VLM -> semantic objects/triples -> scene graph + knowledge memory
memory graph -> native reasoning -> NAVIGATE / EXPLORE / VERIFY
```

GT depth、GT c2w 和 LiDAR 用于离线研究：测量深度尺度漂移、点云误差和局部子地图稳定性。它们不是 RGB-only agent 的运行时前提。

stable submap 会带 confidence、overlap residual 和 provenance 写入 memory。低置信度或相互冲突的证据应促使 agent 重新观察或执行 `VERIFY`，而不是把所有几何关系当作绝对真相。